# Mawārith Analytics — Project 4
## Mathematical Validation of Qur'anic Inheritance Fractions
**Method:** Number theory (LCM/Asl al-Masala), linear algebra (matrix A·x=b), exact rational arithmetic  
**Dataset:** 20 PCIED cases + 5,000 simulation records  
**Institute:** Elbaseety Institute® | Mawārith Analytics  
**Source Data:** PCIED 2023 — Alhikma University, Ilorin

This notebook proves, using exact mathematics, that the Fiqh al-Mawārith engine satisfies five core constraints — and documents two genuine bugs the validation process discovered and fixed.


In [ ]:
import sys; sys.path.insert(0,'.')
from validation_engine import run_full_validation
results = run_full_validation()

## Bugs Discovered & Fixed by This Validation

The validation suite was not a formality — it caught **two real defects** in the v2 engine that had silently produced incorrect distributions in prior simulation runs:

### Bug 1: Son-only Asaba branch missing
When a **son was present without any daughter**, the engine's `if/elif` chain for residuary (Asaba) assignment had no matching branch — the son received **0** instead of the full residue. This affected 363 of 5,000 simulated cases (7.3%).

**Fix:** Added explicit `elif has_son and not has_daughter` branch assigning full residue.

### Bug 2: Spouse-only Radd fallback missing
When a **spouse was the sole heir** (no children, parents, or siblings), the Radd logic correctly excludes spouses from redistribution among *other* Furud heirs — but had no fallback for when *no* other heir exists. The spouse was capped at their fixed 1/4, 1/8, 1/2 share with the remainder unassigned. This affected 163 of 5,000 cases (3.3%).

**Fix:** Added fallback — when spouse is the only heir, full residue reverts to spouse via Radd.

**Combined impact:** 526/5,000 cases (10.5%) failed the sum=1 constraint before these fixes. After both fixes: **0/5,000 failures.**


## 1. Validation Pillars — PCIED 20 Cases

In [ ]:
for c in results['pcied_validation']:
    status = "✓" if c['overall_pass'] else "✗"
    print(f"Case {c['case_id']:2d} {status}  Sum={c['sum_total']:>6}  Asl_valid={c['asl_valid']}  Matrix_valid={c['matrix_valid']}  Quranic_valid={c['quranic_valid']}")
print()
s = results['pcied_summary']
print("ALL PILLARS:", all(s.values()))

## 2. Number Theory — Asl al-Masala Verification

In [ ]:
from IPython.display import Image, display
display(Image('fig2_asl_matrix.png'))

### Why LCM Matters

The Asl al-Masala (base) must be the **least common multiple** of all denominators in the share fractions — not just *any* common multiple. Using a non-minimal base would still produce mathematically correct ratios but would violate the classical Fiqh convention of using the smallest possible whole-number base.

We verify this for every case using Python's `math.gcd` to compute `lcm(a,b) = a*b // gcd(a,b)`, then confirm it matches the engine's reported base exactly.


## 3. Linear Algebra — Matrix Formulation A·x = b

In [ ]:
import sys; sys.path.insert(0,'.')
from mawaarith_engine_v2 import MawaarithEngine, PCIED_CASES
from validation_engine import build_share_matrix, verify_matrix_system
from fractions import Fraction

engine = MawaarithEngine()
case5 = PCIED_CASES[4]  # Khadijat Saheed - the 'Awl case
r = engine.compute(case5)
A, b, labels = build_share_matrix(r.shares_table)

print("Case 5 (Khadijat Saheed) — Matrix System")
print(f"Heirs: {labels}")
print(f"Matrix A ({len(A)}x{len(A[0])}):")
for row in A:
    print("  ", [str(x) for x in row])
print(f"Vector b: {[str(x) for x in b]}")

x = [Fraction(row["share_fraction"]) for row in r.shares_table
     if row["status"]=="Inherits" and row["share_fraction"] not in ("—","0")]
valid, residuals = verify_matrix_system(A, x, b)
print(f"\nA·x = b verified exactly: {valid}")
print(f"Residuals: {[str(x) for x in residuals]}")

## 4. Bug Discovery Visualisation

In [ ]:
display(Image('fig1_validation_summary.png'))

## 5. Qur'anic Fraction Set Membership

Every fixed-share (Furud) heir's base fraction — before any 'Awl or Radd adjustment — must belong to the canonical set explicitly mentioned in Surah An-Nisa (4:11-12, 176):

$$\{1/2, 1/3, 1/4, 1/6, 1/8, 2/3\}$$

We verify this structurally for every heir role in every PCIED case. All 20 cases pass — confirming the engine never invents a fraction outside the Qur'anic set as a base Furud share.


## 6. Conclusion

| Validation Pillar | PCIED (20 cases) | Simulation (5,000 cases) |
|---|:---:|:---:|
| Sum = 1 (exact) | 20/20 ✓ | 5,000/5,000 ✓ (post-fix) |
| Non-negativity | 20/20 ✓ | 5,000/5,000 ✓ |
| Asl al-Masala minimal | 20/20 ✓ | — |
| Matrix A·x=b solvable | 20/20 ✓ | — |
| Qur'anic fraction membership | 20/20 ✓ | — |

This validation exercise demonstrates the value of formal mathematical proof in software engineering for religious/legal computation: **the bugs were invisible in normal testing** (PCIED's 20 cases never exercised the son-only or spouse-only edge cases) but were caught immediately once we required the engine to satisfy `sum(shares) == 1` across 5,000 diverse scenarios.
